# Reproduce main paper tables and figures

This notebook loads the CSVs and PNGs produced by
`reproduce/reproduce_main_tables.py` and renders the tables and figures
that appear in the NeurIPS 2026 E&D submission *Calibration-Constrained
Diagnostics for Evaluating LLM Self-Evaluation*.

## Setup

1. `conda activate selfeval` (or `pip install -r ../../requirements.txt`)
2. Run `python ../reproduce_main_tables.py` once to populate
   `reproduce/outputs/`. The first run takes a few minutes (cached
   afterwards).
3. Run the cells below.

In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import Image, display, Markdown

ROOT = Path('..').resolve().parent  # selfeval_diagnostics/
OUT = ROOT / 'reproduce' / 'outputs'

if not any(OUT.glob('table1_*.csv')):
    print('reproduce/outputs/ is empty -- running reproduce_main_tables.py ...')
    subprocess.check_call([sys.executable, str(ROOT / 'reproduce' / 'reproduce_main_tables.py')])

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

sorted([p.name for p in OUT.iterdir()])

## Table 1 -- Diagnostic decomposition across models (cross-dataset aggregate)

Average of Math-360 / TruthfulQA / CommonsenseQA. Columns: PVC-VUS, C-PVC-VUS, Gap, PM-C-PVC-VUS, CalibError, SEA. This reproduces Table 1 of the paper.

In [ ]:
t1 = pd.read_csv(OUT / 'table1_cross_dataset_diagnostic_decomposition.csv')
display(t1)

## Table 2 -- Scale extension and ranking reversal

Six models side by side with ECE / Brier / PVC-VUS / C-PVC-VUS / Gap / PM-C-PVC-VUS. The ranking reversal headline: the ECE/Brier-best model (JiuZhang3.0-7B) has near-zero PM-C-PVC-VUS, while Qwen2.5-32B-Instruct is worse by ECE but dominates on PM-C-PVC-VUS.

In [ ]:
t2 = pd.read_csv(OUT / 'table2_scale_and_ranking_reversal.csv')
display(t2)

## Figure 2 -- Combined calibration / PVC plot

Two panels of the paper's Figure 2:

- (a) Number of gamma-shattered categories vs gamma, one curve per model.
- (b) Scatter of PVC-VUS vs C-PVC-VUS; the diagonal is perfect calibration.

In [ ]:
display(Markdown('**(a) PVC across \u03b3**'))
display(Image(filename=str(OUT / 'cross_dataset_average_pvc_plot.png')))

In [ ]:
display(Markdown('**(b) PVC-VUS vs C-PVC-VUS**'))
display(Image(filename=str(OUT / 'cross_domain_pvc_cpvc_comparison.png')))

## Figure 3 -- 3-D C-PVC surfaces (4x3 grid of eleven 7-8B models)

In [ ]:
display(Image(filename=str(OUT / 'cross_dataset_average_cpvc_3d_grid.png')))

## Appendix Q -- Per-dataset breakdown

Paper Appendix Table: one row per (model, dataset) with the full nine-column schema including PM-PVC-VUS and PM-SC-VUS.

In [ ]:
perds = pd.read_csv(OUT / 'appendix_per_dataset_breakdown.csv')
display(perds)

## Appendix R -- MATH-500 robustness check

In [ ]:
m500 = pd.read_csv(OUT / 'appendix_math500_table.csv')
display(m500)

## Appendix T -- Judge correlation

Pearson correlation and agreement rate between target models' self-selection and each of the three reference judges on Math-360.

In [ ]:
jc = pd.read_csv(OUT / 'appendix_judge_correlation.csv')
display(jc)

## Recovering exact paper numbers

The paper rounds to two decimals. A few spot checks:

In [ ]:
checks = {
    'JiuZhang3.0-7B Gap (Table 1)': ('table1', 'JiuZhang3.0-7B', 'Gap', 0.95),
    's1.1-7B PVC-VUS (Table 1)':    ('table1', 's1.1-7B', 'PVC-VUS', 5.83),
    'Qwen2.5-7B-Instruct PM-C-PVC-VUS (Table 1)': ('table1', 'Qwen2.5-7B-Instruct', 'PM-C-PVC-VUS', 0.17),
}
for label, (tab, model, col, expected) in checks.items():
    df = {'table1': t1}[tab]
    actual = float(df.loc[df['Model'] == model, col].iloc[0])
    print(f'{label}: computed={actual:.4f}  paper={expected}  match={abs(actual - expected) < 0.02}')